---

# Principal Component Analysis (PCA) 

**Principal Component Analysis** (PCA) is used in exploratory data analysis and for multidimensionality reduction. The main idea is to project data points onto only the first few principal components to obtain lower-dimensional data while preserving as much of the data's variation as possible. In other words, using PCA we remove the redundant and highly-correlated data and keep only the most significant data features for further analysis.

The first principal component is defined as a direction that maximizes variance of the projected data, the second principal component is a direction orthogonal to the first that is the next to maximize variance, and so on. It can be proved that the principal components are the eigenvectors of the covariance matrix, computed either by eigendecomposition of the covariance matrix or by the **Singular Value Decomposition (SVD)** of the data matrix.


---

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()
plt.rcParams["figure.figsize"] = (10, 8)

from sklearn.datasets import load_wine
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

---

## 2. Load the Wine Dataset

The Wine dataset contains **178 samples** described by **13 chemical features** belonging to one of **3 cultivar classes**. All 13 features are numeric and measured on very different scales — exactly the scenario where PCA is most valuable.

---

In [ ]:
data = load_wine()

X_df = pd.DataFrame(data.data, columns=data.feature_names)
y    = data.target

target_names  = list(data.target_names)
feature_names = list(data.feature_names)

print("Feature matrix shape:", X_df.shape)
print("Target shape        :", y.shape)
print("Classes             :", target_names)
print("\nFeatures:")
for f in feature_names:
    print(f"  {f}")

X_df.head()

In [ ]:
X_df.describe().T

---

## 3. Step 1 — Standardise (Centre and Scale) the Data

The steps in PCA are:

**Step 1: Standardise (centre and scale) the data.**

To centre the data we replace each value $x$ by its deviation from the column mean:
$$x \leftarrow x - \bar{x}$$

Because the 13 wine features span very different ranges (e.g. alcohol ≈ 11–15 vs. proline ≈ 278–1680), we also divide by the standard deviation to obtain *z-scores*:
$$z = \frac{x - \bar{x}}{\sigma}$$

This ensures PCA is not dominated by features with large numerical ranges. We then form the centred matrix $A$.

---

In [ ]:
X = X_df.to_numpy()

# Centre only (subtract column means) — used for the manual SVD below
A = X - X.mean(axis=0)

# Centre AND scale (z-scores) — used for sklearn PCA and downstream tasks
scaled_X = preprocessing.scale(X)

print("Centred matrix A:")
print(f"  Shape     : {A.shape}")
print(f"  Col means : {A.mean(axis=0).round(6)}   (should all be ≈ 0)")
print()
print("Scaled matrix (z-scores):")
print(f"  Col means : {scaled_X.mean(axis=0).round(6)}")
print(f"  Col stds  : {scaled_X.std(axis=0).round(6)}")

---

## 4. Step 2 — Compute the Covariance Matrix

**Step 2: Compute the covariance (or correlation) matrix.**

$$S = \frac{1}{n-1} A A^\top$$

If working with only centred data, $S$ is the **covariance matrix**. If working with scaled (z-score) data, $S$ becomes the **correlation matrix**. Positive off-diagonal entries indicate features that increase together; negative entries indicate an inverse relationship.

---

In [ ]:
n = A.shape[0]
S = (A.T @ A) / (n - 1)   # covariance matrix: 13 × 13

print("Covariance matrix shape:", S.shape)

# Visualise
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(S, cmap="coolwarm")
ax.set_xticks(range(13)); ax.set_yticks(range(13))
ax.set_xticklabels(feature_names, rotation=90, fontsize=7)
ax.set_yticklabels(feature_names, fontsize=7)
ax.set_title("Covariance Matrix — Wine Dataset", fontsize=15)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---

## 5. Step 3 — SVD and Principal Components (from Scratch)

**Step 3: Find the eigenvalues and orthonormal eigenvectors of $S$.**

These eigenvectors are the columns of $\mathbf{U}$ in the Singular Value Decomposition of $A$:

$$A = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^\top$$

The rows of $\mathbf{V}^\top$ (columns of $\mathbf{V}$) are the **principal component directions** — the axes of maximum variance in the data.

---

In [ ]:
U, sigma, Vt = np.linalg.svd(A, full_matrices=False)

print(f"np.shape(U)     = {np.shape(U)}")
print(f"np.shape(sigma) = {np.shape(sigma)}")
print(f"np.shape(Vt)    = {np.shape(Vt)} \n")

# Verify reconstruction: A == U * diag(sigma) * Vt
sigma_mat = np.diag(sigma)
print(f"A == U * sigma_mat * Vt: {np.allclose(A, U @ sigma_mat @ Vt)}")

In [ ]:
# Extract all 13 principal component directions from rows of Vt
PC_directions = Vt.T    # shape (13, 13); each column is one PC direction

PC1  = Vt.T[:, 0]
PC2  = Vt.T[:, 1]
PC3  = Vt.T[:, 2]

print("PC1 direction (loading vector):")
for fname, loading in zip(feature_names, PC1):
    print(f"  {fname:<35} {loading:+.4f}")

---

## 6. Step 4 — Reduce Dimension & Project onto PC1 and PC2

**Step 4 & 5: Find the principal components and reduce the dimension of the data.**

We project all data points (rows of $A$) onto the first two principal component directions:

$$X_{2D} = A \cdot [\mathbf{PC}_1 \;\; \mathbf{PC}_2]$$

The $i$-th PC explains
$$\frac{\sigma_i^2}{\sigma_1^2 + \cdots + \sigma_m^2}$$
of the total variance.

---

In [ ]:
# Projection matrix: first 2 PC directions
W2   = Vt.T[:, :2]          # (13, 2)
X_2D = A @ W2                # (178, 2)

print("Projected data shape:", X_2D.shape)

# Variance explained by each PC
var_explained = (sigma**2) / np.sum(sigma**2)
print("\nVariance explained per PC:")
for i, v in enumerate(var_explained, 1):
    print(f"  PC{i:>2}: {v:.4f}  ({v*100:.2f}%)")

In [ ]:
def cultivar_color(label):
    palette = ["red", "magenta", "lightseagreen"]
    return palette[label]

c = [cultivar_color(label) for label in y]

plt.figure(figsize=(10, 8))
plt.scatter(X_2D[:, 0], X_2D[:, 1], c=c, edgecolors="k", s=60, alpha=0.85)
plt.xlabel("First Principal Component", fontsize=15)
plt.ylabel("Second Principal Component", fontsize=15)
plt.title("PCA from Scratch — Wine Dataset (PC1 vs PC2)", fontsize=16)

for cls, name in enumerate(target_names):
    plt.scatter([], [], color=cultivar_color(cls), label=name, edgecolors="k", s=60)
plt.legend(fontsize=11)
plt.show()

---

## 7. Step 5 — Scree Plot (sklearn PCA)

We now repeat the analysis using `sklearn.decomposition.PCA` on the fully **scaled** (z-score) data and confirm our manual results.

A **Scree Plot** graphs the percentage of total variance explained by each principal component. We look for an "elbow" — the point beyond which additional components contribute little extra information.

---

In [ ]:
pca = PCA()
pca.fit(scaled_X)

print(f"pca.explained_variance_ratio_ = {pca.explained_variance_ratio_.round(4)}")

per_var = np.round(pca.explained_variance_ratio_ * 100, 2)
print(f"\nper_var = {per_var}")

In [ ]:
labels = [f"PC{i}" for i in range(1, 14)]
cumulative = np.cumsum(per_var)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Scree plot
axes[0].bar(range(1, 14), per_var, tick_label=labels, color="steelblue", edgecolor="black")
axes[0].set_xlabel("Principal Component", fontsize=14)
axes[0].set_ylabel("Percentage of Variation (%)", fontsize=14)
axes[0].set_title("Scree Plot — Wine Dataset", fontsize=15)

# Cumulative variance
axes[1].plot(range(1, 14), cumulative, marker="o", color="darkorange")
axes[1].axhline(90, linestyle="--", color="red", label="90% threshold")
axes[1].axhline(95, linestyle="--", color="purple", label="95% threshold")
axes[1].set_xlabel("Number of Principal Components", fontsize=14)
axes[1].set_ylabel("Cumulative Variance Explained (%)", fontsize=14)
axes[1].set_title("Cumulative Explained Variance", fontsize=15)
axes[1].set_xticks(range(1, 14))
axes[1].legend(fontsize=11)

plt.suptitle("Explained Variance — Wine Dataset PCA", fontsize=17)
plt.tight_layout()
plt.show()

print("Cumulative variance explained:")
for i, (pv, cv) in enumerate(zip(per_var, cumulative), 1):
    print(f"  PC{i:>2}: {pv:6.2f}%   cumulative: {cv:6.2f}%")

---

## 8. Loading Scores — How Each PC is a Linear Combination of Features

The **loading scores** stored in `pca.components_` tell us exactly how each principal component is constructed as a linear combination of the original 13 features. Large absolute values indicate features that drive that component.

For example:
$$PC_1 = w_1 \cdot \text{alcohol} + w_2 \cdot \text{malic\_acid} + \cdots + w_{13} \cdot \text{proline}$$

---

In [ ]:
# Loading scores: rows = PCs, columns = features
loadings_df = pd.DataFrame(
    data    = pca.components_.T,   # transpose so rows = features, cols = PCs
    index   = feature_names,
    columns = labels
)

print("Loading scores (features × PCs):")
loadings_df.round(4)

In [ ]:
# Visualise loading scores for PC1 and PC2
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, pc_col, color, title in [
    (axes[0], "PC1", "steelblue",   "PC1 Loading Scores"),
    (axes[1], "PC2", "darkorange",  "PC2 Loading Scores"),
]:
    sorted_df = loadings_df[pc_col].sort_values()
    ax.barh(sorted_df.index, sorted_df.values, color=color, edgecolor="black")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Loading Score", fontsize=13)
    ax.set_title(title, fontsize=14)

plt.suptitle("Feature Contributions to PC1 and PC2", fontsize=16)
plt.tight_layout()
plt.show()

---

## 9. Project Data & Build PCA DataFrame

We use `pca.transform` to express every sample in the new PC coordinate system, then build a tidy DataFrame for plotting.

---

In [ ]:
pca_data = pca.transform(scaled_X)

pca_df = pd.DataFrame(pca_data, columns=labels)
print("PCA-transformed data shape:", pca_df.shape)
pca_df.head()

In [ ]:
# Add true cultivar labels
pca_df["cultivar"] = [target_names[i] for i in y]
pca_df["color"]    = [cultivar_color(i) for i in y]
pca_df

---

## 10. Two-Component PCA Plot (sklearn)

---

In [ ]:
colors_list = ["red", "magenta", "lightseagreen"]

plt.figure(figsize=(10, 8))
for target, color in zip(target_names, colors_list):
    temp_df = pca_df[pca_df["cultivar"] == target]
    plt.scatter(temp_df["PC1"], temp_df["PC2"],
                c=color, label=target, edgecolors="k", s=60, alpha=0.85)

plt.xlabel(f"PC1 ({per_var[0]:.2f}% variance)", fontsize=15)
plt.ylabel(f"PC2 ({per_var[1]:.2f}% variance)", fontsize=15)
plt.title("Two-Component PCA — Wine Dataset", fontsize=18)
plt.legend(fontsize=12)
plt.show()

In [ ]:
# Three-component PCA (3D)
fig = plt.figure(figsize=(12, 9))
ax  = fig.add_subplot(111, projection="3d")

for target, color in zip(target_names, colors_list):
    mask = pca_df["cultivar"] == target
    ax.scatter(pca_df.loc[mask, "PC1"],
               pca_df.loc[mask, "PC2"],
               pca_df.loc[mask, "PC3"],
               c=color, label=target, s=50, alpha=0.85)

ax.set_xlabel(f"PC1 ({per_var[0]:.1f}%)", fontsize=11)
ax.set_ylabel(f"PC2 ({per_var[1]:.1f}%)", fontsize=11)
ax.set_zlabel(f"PC3 ({per_var[2]:.1f}%)", fontsize=11)
ax.set_title("Three-Component PCA — Wine Dataset", fontsize=15)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---

## 11. Verify: Manual SVD vs. sklearn PCA

As a sanity check we confirm that the projection produced by our manual SVD and by `sklearn` agree (up to possible sign flips, which are arbitrary in PCA).

---

In [ ]:
# sklearn projects scaled_X; our manual SVD projected centred-only A.
# Redo manual on scaled_X for a fair comparison.
A_scaled = scaled_X - scaled_X.mean(axis=0)  # already zero mean, but be explicit
U_s, sigma_s, Vt_s = np.linalg.svd(A_scaled, full_matrices=False)
X_2D_manual = A_scaled @ Vt_s.T[:, :2]

skl_2D = pca.transform(scaled_X)[:, :2]

# Sign flip: sklearn may flip the sign of a PC — both are equally valid
# Check column by column
for col in range(2):
    if not np.allclose(X_2D_manual[:, col], skl_2D[:, col], atol=1e-8):
        if np.allclose(X_2D_manual[:, col], -skl_2D[:, col], atol=1e-8):
            print(f"PC{col+1}: manual and sklearn agree up to a sign flip (both valid).")
        else:
            print(f"PC{col+1}: MISMATCH — investigate further.")
    else:
        print(f"PC{col+1}: manual and sklearn are identical.")

---

## 12. PCA as a Preprocessing Step — Impact on Classification

One of the most practical uses of PCA is as a **preprocessing step** before a supervised classifier. Reducing 13 features down to a small number of PCs can:
- Remove noise and multicollinearity.
- Speed up training.
- Sometimes improve generalisation.

We compare a Random Forest trained on the full 13 features against one trained on the top $k$ PCs.

---

In [ ]:
# Train / test split on scaled data
X_train_full, X_test_full, y_train, y_test = train_test_split(
    scaled_X, y, test_size=0.33, stratify=y, random_state=42
)

# Baseline: Random Forest on all 13 features
rf_full = RandomForestClassifier(n_estimators=200, random_state=42)
rf_full.fit(X_train_full, y_train)
acc_full = accuracy_score(y_test, rf_full.predict(X_test_full))
print(f"Random Forest (13 features) test accuracy: {acc_full:.4f}")

In [ ]:
# PCA + Random Forest for k = 1..13
k_range  = range(1, 14)
k_accs   = []

for k in k_range:
    pca_k = PCA(n_components=k, random_state=42)
    X_tr_k = pca_k.fit_transform(X_train_full)
    X_te_k = pca_k.transform(X_test_full)

    rf_k = RandomForestClassifier(n_estimators=200, random_state=42)
    rf_k.fit(X_tr_k, y_train)
    k_accs.append(accuracy_score(y_test, rf_k.predict(X_te_k)))

print(f"{'k PCs':>6} | {'Accuracy':>10}")
print("-" * 20)
for k, acc in zip(k_range, k_accs):
    print(f"{k:>6} | {acc:>10.4f}")

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(list(k_range), k_accs, marker="o", color="steelblue", label="PCA + RF")
plt.axhline(acc_full, linestyle="--", color="red",
            label=f"Full features baseline ({acc_full:.4f})")
plt.xlabel("Number of Principal Components", fontsize=13)
plt.ylabel("Test Accuracy", fontsize=13)
plt.title("Random Forest Accuracy vs. Number of PCA Components", fontsize=15)
plt.xticks(list(k_range))
plt.legend(fontsize=11)
plt.show()

In [ ]:
# Best PCA-reduced model
best_k   = list(k_range)[k_accs.index(max(k_accs))]
best_acc = max(k_accs)
print(f"Best k = {best_k}  (accuracy = {best_acc:.4f})")
print(f"Variance captured by {best_k} PCs: {cumulative[best_k-1]:.2f}%")

pca_best = PCA(n_components=best_k, random_state=42)
X_tr_best = pca_best.fit_transform(X_train_full)
X_te_best = pca_best.transform(X_test_full)

rf_best = RandomForestClassifier(n_estimators=200, random_state=42)
rf_best.fit(X_tr_best, y_train)
y_pred_best = rf_best.predict(X_te_best)

print(f"\nClassification Report (RF + {best_k} PCs):")
print(classification_report(y_test, y_pred_best, target_names=target_names))